# betting-ad-blocker — local training notebook

Mirrors `scripts/train.py` cell-by-cell for users who prefer a notebook workflow. Reads the same `config.yaml` and writes the same outputs (`models/best.pt`, `runs/segment/...`) as the CLI script, so you can freely switch between the two.

## 1. Environment check

In [ ]:
!nvidia-smi

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

import torch
from common import load_config

cfg = load_config(REPO_ROOT / "config.yaml")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
cfg["train"]

## 2. Dataset sanity check

Renders a handful of assembled training images with their polygons drawn, so you can visually confirm labels line up before burning GPU hours on a bad dataset.

In [ ]:
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np

dataset_dir = cfg["paths"]["dataset"]
class_names = {0: "person", 1: "betting_board", 2: "betting_overlay"}
class_colors = {0: (255, 0, 0), 1: (0, 200, 0), 2: (0, 128, 255)}

img_dir = dataset_dir / "images" / "train"
images = sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png"))
assert images, f"No training images found in {img_dir} — run assemble_dataset.py first."

sample = random.sample(images, k=min(6, len(images)))
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, img_path in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = dataset_dir / "labels" / "train" / f"{img_path.stem}.txt"
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if len(parts) < 7:
                continue
            cls = int(parts[0])
            coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
            coords[:, 0] *= w
            coords[:, 1] *= h
            color = class_colors.get(cls, (255, 255, 255))
            cv2.polylines(img, [coords.astype(np.int32)], True, color, 2)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. Train

Set `WEIGHTS = REPO_ROOT / "models" / "best.pt"` and lower `epochs`/`lr0` (see `config.yaml`'s `finetune_*` values) to fine-tune from the current model instead of training fresh from COCO.

**⚠️ If real-video testing shows detection confidence around 0.4-0.5 on a large, obvious board, the model is undertrained** — this is expected with a small real-annotation set. `scripts/process_video.py`'s panel-persistence tracker (custom ByteTrack + camera-motion propagation, see `trackers/betting_bytetrack.yaml`) is specifically designed to make hiding robust *despite* this, but it cannot fix low recall on its own. For production-quality confidence, annotate 300+ real frames — whole ad strips as ONE `betting_board` polygon each (see the labeling convention in `README.md` §2), covering all camera angles **including motion-blurred pan frames** — then run a full 100-epoch retrain (`train.py`, not just a short fine-tune).

Because of this, `inference.conf` in `config.yaml` is kept deliberately low (0.10) — it only feeds the tracker, not the renderer, and for this product missing an ad (false negative) is worse than an extra low-confidence detection the tracker discards on its own (recall matters more than precision here).

In [ ]:
from ultralytics import YOLO

tcfg = cfg["train"]
WEIGHTS = None  # e.g. REPO_ROOT / "models" / "best.pt" to fine-tune instead of training fresh

start_from = str(WEIGHTS) if WEIGHTS else tcfg["model"]
model = YOLO(start_from)

results = model.train(
    data=str(cfg["paths"]["data_yaml"]),
    epochs=tcfg["finetune_epochs"] if WEIGHTS else tcfg["epochs"],
    imgsz=tcfg["imgsz"],
    batch=tcfg["batch"],
    device=tcfg["device"],
    amp=tcfg["amp"],
    workers=tcfg["workers"],
    cache=tcfg["cache"],
    patience=tcfg["patience"],
    project=tcfg["project"],
    name=tcfg["name"],
    lr0=tcfg["finetune_lr0"] if WEIGHTS else None,
)

In [ ]:
import shutil

run_dir = results.save_dir
best_ckpt = run_dir / "weights" / "best.pt"
dest = cfg["paths"]["models"] / "best.pt"
cfg["paths"]["models"].mkdir(parents=True, exist_ok=True)
shutil.copy2(best_ckpt, dest)
print(f"Copied {best_ckpt} -> {dest}")

## 4. Results / curves

In [ ]:
from PIL import Image

results_png = run_dir / "results.png"
if results_png.exists():
    display(Image.open(results_png))
else:
    print(f"No results.png yet at {results_png}")

## 5. Quick inference sanity check

In [ ]:
val_images = sorted((dataset_dir / "images" / "val").glob("*.jpg"))
if val_images:
    trained = YOLO(str(dest))
    preds = trained.predict(source=str(random.choice(val_images)), conf=cfg["inference"]["conf"])
    annotated = preds[0].plot()[:, :, ::-1]
    plt.figure(figsize=(10, 6))
    plt.imshow(annotated)
    plt.axis("off")
    plt.show()
else:
    print("No val images to run a quick inference check on.")